# DFT Pseudopotentials

This notebook introduces the Milestone 4 ion model. We parse real UPF and GTH reference files, inspect their local potentials, and run small local-pseudopotential SCF calculations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import DFTSystem, DiracExchange, Ion, IonCollection, SCFConfig, read_gth, read_upf, run_scf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

UPF supplies a tabulated radial local potential. GTH supplies a compact analytic local potential. Nonlocal projector metadata is parsed and stored, but this milestone applies only the local part.

In [ ]:
upf = read_upf(ROOT / "vendors/quantum-espresso/pseudo/Si_r.upf")
gth = read_gth(ROOT / "vendors/quantum-espresso/pseudo/H-q1.gth", element="H")
{
    "upf": (upf.element, upf.valence_charge, upf.local_grid.size, len(upf.nonlocal_projectors)),
    "gth": (gth.element, gth.valence_charge, gth.gth_rloc, len(gth.nonlocal_projectors)),
}

In [ ]:
r = np.linspace(0.02, 4.0, 300)
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.plot(r, upf.local_potential(r), label="Si UPF local")
ax.plot(r, gth.local_potential(r), label="H GTH local")
ax.set_xlabel("r / bohr")
ax.set_ylabel("V_local / Hartree")
ax.set_title("local pseudopotential radial profiles")
ax.legend();

The neutral electron count can be inferred from the valence charge. These runs are numerical smoke checks, not chemically validated calculations.

In [ ]:
config = SCFConfig(max_iterations=1, solver="dense", seed=17)
systems = {
    "gth": DFTSystem(
        cell=(6.0, 6.0, 6.0),
        grid_shape=(4, 4, 4),
        ions=IonCollection([Ion("H", (3.0, 3.0, 3.0), gth)]),
    ),
    "upf": DFTSystem(
        cell=(8.0, 8.0, 8.0),
        grid_shape=(4, 4, 4),
        ions=IonCollection([Ion("Si", (4.0, 4.0, 4.0), upf)]),
    ),
}
results = {name: run_scf(system, config=config, xc_functional=DiracExchange()) for name, system in systems.items()}
{name: {key: result.to_dict()[key] for key in ["total_energy", "electron_count", "pseudopotential_format", "nonlocal_available"]} for name, result in results.items()}